# AI Reliability and Release Engineering

Operate a staged model release using service objectives, trace diagnosis, and an incident exercise.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · Four timeouts in a small rollout

Olivia is on call for an assistant that summarizes support tickets. A candidate version is handling a small cohort while the stable version serves everyone else. Several pilot requests are late and some need the manual queue. The rest of the system looks healthy.

The workplace scenario is fictional.


## Define and observe reliability


### Lecture 01 · Service objectives and error-budget burn

Define a good request and calculate reliability and budget burn over a bounded window.

#### A user can receive a response and still lose time

A late summary eventually arrives, after an agent already opened a second ticket to work around the delay. Transport success stays green.

Define a good outcome in terms of valid and timely work. Use the cohort denominator and the agreed objective to estimate burn. Keep the observation window and sample size attached to the number.

#### How it works

A service objective uses outcomes a user cares about, such as a valid, grounded result delivered under a latency bound. The error budget is the allowed fraction of bad requests. Burn rate compares the observed bad fraction with that allowance.

##### Count a good request as the caller experiences it

A service-level indicator should define a usable result. For an assistant, that could require a valid answer or useful handoff within a deadline. A transport success with a late, malformed, or unsupported result might fail this definition. State the eligible population and observation window.

An objective of 99% good results allows 1% bad results in the chosen window. If the measured bad rate is 5%, the budget is being consumed at five times the allowed rate for that window. Burn expresses the pace of failure relative to the target, rather than the total incident impact on its own.

Check the cohort denominator and sample size. A small sample can be enough to reveal a severe problem, even though it gives a noisy estimate of the true rate. Decide when to alert, when to gather more evidence, and when a critical failure independently requires pausing.

#### Worked example

With a 99% objective, a 5% bad rate burns error budget at five times the sustainable rate.

#### Try it

Compute objectives for the fixture cohorts. Change the latency threshold and decide which failures count.


In [ ]:
from dataclasses import dataclass
from statistics import median


@dataclass(frozen=True)
class Trace:
    cohort: str
    version: str
    stages_ms: tuple[tuple[str, int], ...]
    valid: bool
    failed_at: str | None = None
    cost: float = 0.002

    @property
    def elapsed(self):
        return sum(value for _, value in self.stages_ms)


def trace(cohort, version, bad=False):
    return Trace(cohort, version, (("retrieval", 20), ("generation", 900 if bad else 180), ("validation", 15)),
                 not bad, "generation" if bad else None)


stable = [trace("control", "v1") for _ in range(20)]
healthy = [trace("pilot", "v2") for _ in range(20)]
unhealthy = [trace("pilot", "v2", bad=i < 4) for i in range(20)]


def objectives(rows, latency_limit=350, target=0.99):
    good = sum(row.valid and row.elapsed <= latency_limit for row in rows)
    bad_fraction = (len(rows) - good) / len(rows)
    return {"n": len(rows), "good": good, "burn": bad_fraction / (1 - target),
            "median_ms": median(row.elapsed for row in rows), "mean_cost": sum(row.cost for row in rows) / len(rows)}


assert objectives(stable)["burn"] == 0
assert abs(objectives(unhealthy)["burn"] - 20) < 1e-8


**Compare your result**

A successful transport can still be a bad outcome if the result is invalid or late. State the denominator and report low sample counts.


### Lecture 02 · Stage traces and failure localization

Use structured events to locate the earliest divergence between candidate and baseline.

#### Start at the earliest stage that differs

Retrieval and validation look similar across versions. Generation in the pilot shows a long wait and failed calls.

Compare stage traces for stable and pilot requests with similar inputs. Separate observed divergence from the proposed cause. The operator needs a bounded reason to disable the candidate while deeper diagnosis continues.

#### How it works

Record the first failing stage and bounded latency signals for retrieval, generation, and validation. Separate infrastructure failures from poor answer quality and group by version and cohort.

##### Follow a trace from user outcome to the earliest divergence

A stage trace connects the user's result with what happened in routing, retrieval, generation, and validation. Record timings, versions, outcomes, and a request identity. Keep sensitive content out of telemetry when a category or identifier is enough. A low-level exception alone rarely tells an operator what the user experienced.

Compare an affected request with a similar stable one. Start at the earliest point where their behavior differs. Slow generation might be the visible issue, while the deeper cause could be an overly long context, a dependency delay, or a new route. Distinguish those observations from hypotheses.

Exercise the distinction in the lab by injecting a delay at a known stage. A report that attributes the failure correctly gives the next responder a bounded place to investigate. Include fallback behavior so served results do not hide a failed primary or an unexpected burden on staff.

#### Worked example

Candidate users receive several timeouts during generation. Retrieval is fast for both cohorts, so the trace points investigation to the candidate dependency.

#### Try it

Run the failure aggregation and examine how the diagnosis changes if you move the injected failure to retrieval.


In [ ]:
from collections import Counter


def failure_stages(rows):
    return Counter(row.failed_at for row in rows if row.failed_at)


assert failure_stages(stable) == {}
assert failure_stages(unhealthy) == {"generation": 4}
print("First failing stage", failure_stages(unhealthy).most_common(1))


**Compare your result**

The dominant failing stage should follow the injection while stable traffic stays unaffected.


## Release and recover


### Lecture 03 · Canary cohorts and staged gates

Use predeclared coverage, burn, latency, and cost thresholds to approve or hold a rollout.

#### Predeclare the stop rule for the small cohort

The whole-service error rate appears stable because most requests go to the baseline. The pilot has four failures among twenty requests.

Apply quality, latency, and cost gates to the pilot version with only its own denominator. Keep an explicit minimum sample and critical stop conditions. The aggregate cannot replace cohort evidence.

##### Make the call while the stable version is available

Assume the release owner established a 95% threshold for on-time, valid responses and requires at least 20 candidate requests before deciding on expansion. The pilot records 16 good results in 20 attempts. That's 80% for the pilot. The baseline records 99 good results in 100 attempts. Combining both produces 115 good results in 120 attempts, or about 96%.

The combined number clears the threshold while the candidate fails it. Hold the candidate. Compare the earliest failed stage with comparable baseline requests and verify that the stable version can still serve the pilot users. A small cohort limits exposure. It does not make its missing or late work less important.

Now change the pilot to one failure in 20. The observed rate equals the threshold. Would you expand immediately? Check whether the minimum sample addresses the workload you are about to expose, including any consequential slice missing from the pilot.

#### How it works

Keep stable and candidate traffic separately attributed and compare them over a defined window. A gate checks sample size, the critical failures, and cost before expanding. Report the exact gate that stops the rollout.

##### Protect the canary from misleading averages

Assign a small cohort to the candidate and measure that cohort alongside comparable stable traffic. Record both the attempted and served versions, especially when fallback is enabled. If most requests use the baseline, a healthy aggregate can coexist with a candidate that fails much more often.

Predeclare a primary outcome, minimum observation requirement, and blocking conditions. Consider valid results, latency, fallback burden, and cost per usable task. Confirm that the canary contains the cases the release will serve. Missing evidence may justify holding the current exposure while collecting more data.

Keep the rollback route live and define who can invoke it. In the workbook, compare one passing and one failing cohort. An operator should be able to explain which criterion blocked expansion and what observation would justify resuming, without reverse-engineering the decision from a headline score.

#### Worked example

A cheaper candidate misses the quality and latency gates during a 10% cohort. Pause that version and continue serving stable traffic.

#### Try it

Run a healthy and an unhealthy canary. Relax one gate and inspect which other condition still blocks promotion.


In [ ]:
def gate(rows, sample=20, burn_limit=2, latency_limit=350, cost_limit=0.003):
    if len(rows) < sample:
        return ["coverage"]
    info = objectives(rows)
    limits = (
        ("error_budget", info["burn"], burn_limit), ("latency", info["median_ms"], latency_limit),
        ("cost", info["mean_cost"], cost_limit),
    )
    return [name for name, observed, limit in limits if observed > limit]


assert gate(healthy) == []
assert gate(healthy[:1]) == ["coverage"]
assert gate(unhealthy) == ["error_budget"]
assert gate(unhealthy, burn_limit=21, cost_limit=0.001) == ["cost"]


**Compare your result**

A canary passes only after all required evidence conditions pass. A low-cost result still needs adequate reliability.


### Lecture 04 · Rollback verification and incident review

Restore stable routing, run a smoke check, and assemble a factual incident timeline.

#### Write the timeline while decisions are still clear

The candidate is disabled and the next test request succeeds. Olivia needs to prove that affected users are now served by the stable model and that the queue is recovering.

Check both cohorts, note the observed version and outcome, then record detection, decision, mitigation, and validation in order. Attach one owned follow-up with a concrete verification condition.

#### How it works

After a rollback, verify that new requests reach the expected version and that the service indicators recover. Record detection, decision, mitigation, and validation times. Write follow-up actions with an owner and verification criterion.

##### Close the loop from rollback to user recovery

Restore the prior serving version and verify the effect through a request representative of the affected cohort. Confirm version, result, latency, and whether any manual work remains queued. The rollback can restore new traffic while leaving older work that still needs reconciliation.

Build a timeline while evidence is available. Separate detection, decision, mitigation, verification, and follow-up. Preserve the observed impact and the source of each observation. If the cause is uncertain, label it and assign a way to investigate rather than treating a plausible story as a confirmed explanation.

A useful follow-up has an owner and a verification condition. “Add monitoring” is vague. “Alert when generation wait breaches the candidate gate on the replayed fixture” tells reviewers how to know the change works. Verify that people can still resolve delayed requests, not only that the graph returned to normal.

#### Worked example

The on-call engineer pauses the candidate, confirms stable responses from the affected cohort, and records a follow-up to test dependency failure before the next release.

#### Try it

Run the scripted incident. Extend the report with one owner, one preventable failure, and a due condition.


In [ ]:
class Rollout:
    def __init__(self, baseline, candidate):
        self.baseline = baseline
        self.candidate = candidate

    def route(self, cohort):
        return self.candidate if cohort == "pilot" and self.candidate else self.baseline

    def rollback(self):
        self.candidate = None


rollout = Rollout("v1", "v2")
assert rollout.route("pilot") == "v2"
timeline = [(0, "canary started", "v2"), (5, "error-budget gate blocked", gate(unhealthy))]
rollout.rollback()
timeline.append((6, "candidate disabled", rollout.route("pilot")))
smoke = trace("pilot", rollout.route("pilot"))
assert smoke.version == "v1" and smoke.valid and smoke.elapsed <= 350
timeline.append((7, "smoke checked", {"version": smoke.version, "elapsed_ms": smoke.elapsed}))
assert [minute for minute, _, _ in timeline] == sorted(minute for minute, _, _ in timeline)
print("Incident timeline", timeline)
print("Follow-up: service owner reproduces the dependency timeout in pre-release checks.")


**Compare your result**

The incident review distinguishes measured cause and impact from hypotheses and includes evidence that rollback restored service.


## Course project

Run a staged rollout with an injected dependency failure. Diagnose from traces, roll back, and submit an incident review with follow-up owners.

Specify user-facing service objectives, compute error-budget burn, apply canary gates by cohort, and verify after rollback that the baseline serves successfully.
